In [39]:
df = pd.read_csv(
    "abcd_fbwss01.txt",          # or .csv if you exported via the API
    sep="\t",                    # NDA text exports use tabs
    low_memory=False
)

# 1. Consistent week type
df = df[df["fit_ss_datatype"] == '2']

# # 2. Adequate wear
df = df[df["fit_ss_day_count"] >= '4']




# # 4. Choose one record per child (first week)
# df = df.sort_values("fit_ss_protocol_date").drop_duplicates("src_subject_id")

# # # 5. Domain label
# # df["sleep_domain"] = np.where(df["fit_ss_avg_sleep_period_min"] < 450,
# #                               "short", "long")

In [43]:
df['fit_ss_avg_sleep_period_min'].isna().sum()

35866

In [22]:
df = pd.read_csv(
    "abcd_fbwss01.txt",          # or .csv if you exported via the API
    sep="\t",                    # NDA text exports use tabs
    low_memory=False
)

# ------------------------------------------------------------------
# 2. Minimal QC  -----------------------------------------------------
#    – keep only weeks summarised across both weekdays & weekends
#    – require at least 4 days of valid wear in that week
# ------------------------------------------------------------------
df = df.query("fit_ss_datatype == '2' and fit_ss_day_count >= '4'")

In [23]:
df

,collection_id,abcd_fbwss01_id,dataset_id,subjectkey,src_subject_id,interview_date,interview_age,sex,eventname,fit_ss_protocol_date,...,fit_ss_sleep_avg_light_minutes,fit_ss_sleep_avg_deep_minutes,fit_ss_sleep_avg_rem_minutes,fit_ss_sleep_avg_wake_count,fit_ss_sleep_avg_hr_wake,fit_ss_sleep_avg_hr_light,fit_ss_sleep_avg_hr_deep,fit_ss_sleep_avg_hr_rem,collection_title,study_cohort_name
1,2573,38041,47283,NDAR_INVZNW7Y6R1,NDAR_INVZNW7Y6R1,09/10/2016,116,F,baseline_year_1_arm_1,09/05/2017,...,177,12,33,14,57,58,60,65,Adolescent Brain Cognitive Development Study (...,ABCD 4.0 Data Release
2,2573,38043,47283,NDAR_INVZZZ2ALR6,NDAR_INVZZZ2ALR6,06/15/2017,121,F,baseline_year_1_arm_1,06/15/2017,...,213,117,114,30,70,64,70,65,Adolescent Brain Cognitive Development Study (...,ABCD 4.0 Data Release
3,2573,38048,47283,NDAR_INV06DE9Y0L,NDAR_INV06DE9Y0L,11/28/2017,108,M,baseline_year_1_arm_1,11/28/2017,...,284,82,79,26,86,84,87,87,Adolescent Brain Cognitive Development Study (...,ABCD 4.0 Data Release
4,2573,38055,47283,NDAR_INV0AUBJJJ4,NDAR_INV0AUBJJJ4,10/21/2017,116,F,baseline_year_1_arm_1,10/21/2017,...,259,95,114,31,70,67,68,73,Adolescent Brain Cognitive Development Study (...,ABCD 4.0 Data Release
5,2573,38058,47283,NDAR_INV0AU5R8NA,NDAR_INV0AU5R8NA,08/17/2017,117,F,baseline_year_1_arm_1,08/17/2017,...,294,64,60,26,82,75,77,79,Adolescent Brain Cognitive Development Study (...,ABCD 4.0 Data Release
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35850,2573,73773,47283,NDAR_INVZUKTAC4W,NDAR_INVZUKTAC4W,10/27/2019,132,M,2_year_follow_up_y_arm_1,10/27/2019,...,270,82,89,33,71,67,68,70,Adolescent Brain Cognitive Development Study (...,ABCD 4.0 Data Release
35854,2573,73792,47283,NDAR_INVZ4NNCTMH,NDAR_INVZ4NNCTMH,11/23/2019,150,F,2_year_follow_up_y_arm_1,11/23/2019,...,302,104,90,30,68,65,70,63,Adolescent Brain Cognitive Development Study (...,ABCD 4.0 Data Release
35860,2573,73828,47283,NDAR_INVZVD13ZMG,NDAR_INVZVD13ZMG,12/29/2018,146,F,2_year_follow_up_y_arm_1,12/29/2018,...,255,83,106,33,81,76,78,80,Adolescent Brain Cognitive Development Study (...,ABCD 4.0 Data Release
35862,2573,73848,47283,NDAR_INVZW8G4W5A,NDAR_INVZW8G4W5A,06/22/2019,149,M,2_year_follow_up_y_arm_1,06/22/2019,...,316,118,145,54,65,61,66,66,Adolescent Brain Cognitive Development Study (...,ABCD 4.0 Data Release


In [31]:
df['fit_ss_avg_sleep_period_min'].isna().sum()

8669

In [45]:
df[sleep_sum]

1         222
2        1333
3        3112
4        2808
5        1672
         ... 
35850    2201
35854    2479
35860    2659
35862    2893
35866    2250
Name: fit_ss_sleep_period_minutes, Length: 8669, dtype: object

In [46]:
import pandas as pd
import numpy as np

# ------------------------------------------------------------------
# 1. Load the weekly-summary file you grabbed from NDA
#    (tab-delimited; change path as needed)
# ------------------------------------------------------------------
df = pd.read_csv(
    "abcd_fbwss01.txt",          # or .csv if you exported via the API
    sep="\t",                    # NDA text exports use tabs
    low_memory=False
)

# ------------------------------------------------------------------
# 2. Minimal QC  -----------------------------------------------------
#    – keep only weeks summarised across both weekdays & weekends
#    – require at least 4 days of valid wear in that week
# ------------------------------------------------------------------
df = df.query("fit_ss_datatype == '2' and fit_ss_day_count >= '4'")


sleep_sum   = "fit_ss_sleep_period_minutes"   # weekly total minutes
day_count   = "fit_ss_day_count"              # number of valid days
df[sleep_sum] = pd.to_numeric(df[sleep_sum], errors="coerce")
df[day_count] = pd.to_numeric(df[day_count], errors="coerce")
# … after filtering on fit_ss_datatype == 2 and fit_ss_day_count >= 4 …

# 3-A.  Derive nightly average *for each week* -----------------------
df["sleep_avg_min"] = df[sleep_sum] / df[day_count]

# 3-B.  Then do the weighted aggregation across weeks ---------------
def wmean(group, value, w):
    return np.average(group[value], weights=group[w])

agg = (
    df
    .groupby("src_subject_id")
    .apply(lambda g: pd.Series({
        "sleep_avg_min":   wmean(g, "sleep_avg_min", day_count),
        "total_day_count": g[day_count].sum(),
        "weeks_used":      len(g)
    }))
    .reset_index()
)


# # ------------------------------------------------------------------
# # 3. Weighted average sleep minutes across weeks for each child -----
# #    We weight by `fit_ss_day_count` so that a 6-day week influences
# #    the child’s mean more than a 4-day week.
# # ------------------------------------------------------------------
# cols_keep = [
#     "src_subject_id",
#     "interview_age",
#     "sex"                       # <-- keep any covariates you want
# ]

# sleep_min      = "fit_ss_avg_sleep_period_min"
# weight         = "fit_ss_day_count"

# # helper: weighted mean
# def wmean(group, value, w):
#     return np.average(group[value], weights=group[w])

# agg = (
#     df
#     .groupby("src_subject_id")
#     .apply(lambda g: pd.Series({
#         "sleep_avg_min":   wmean(g, sleep_min, weight),
#         "total_day_count": g[weight].sum(),      # optional diagnostics
#         "weeks_used":      len(g)
#     }))
#     .reset_index()
# )

# # ------------------------------------------------------------------
# # 4. Merge back demographics (first record is fine; they don’t vary)
# # ------------------------------------------------------------------
# first_demo = df.groupby("src_subject_id").first().reset_index()[cols_keep]
# final = first_demo.merge(agg, on="src_subject_id", how="inner")

# # ------------------------------------------------------------------
# # 5. Create Option A domain label (7.5-hour cut-off = 450 min) -------
# # ------------------------------------------------------------------
# final["sleep_domain"] = np.where(final["sleep_avg_min"] < 450,
#                                  "short", "long")

# # ------------------------------------------------------------------
# # 6. Save or inspect
# # ------------------------------------------------------------------
# # final.to_csv("abcd_sleep_domain_per_child.csv", index=False)
# print(final.head())

In [48]:
df['sleep_avg_min'].describe()

count    8669.000000
mean      446.914803
std        54.869392
min        44.400000
25%       423.600000
50%       453.200000
75%       478.000000
max       860.250000
Name: sleep_avg_min, dtype: float64

In [49]:
np.median(df['sleep_avg_min'])

453.2